# Dataset Validation & Operational Data Review

This notebook validates the anonymised operational datasets used for the spare-parts warehouse analysis and automated warehouse transition project.

The objective is to confirm that the main warehouse, inventory, demand, supplier, and replenishment tables are structurally consistent before starting the detailed operational analyses.

In [18]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

part_master = pd.read_csv("../data/part_master.csv")
equipment_models = pd.read_csv("../data/equipment_models.csv")
supplier_master = pd.read_csv("../data/supplier_master.csv")
demand_history = pd.read_csv("../data/demand_history_weekly.csv")
inventory_status = pd.read_csv("../data/inventory_status.csv")
purchase_requisitions = pd.read_csv("../data/purchase_requisitions.csv")
data_dictionary = pd.read_csv("../data/data_dictionary.csv")

if "Week_Start" in demand_history.columns:
    demand_history["Week_Start"] = pd.to_datetime(demand_history["Week_Start"])

if "Requisition_Date" in purchase_requisitions.columns:
    purchase_requisitions["Requisition_Date"] = pd.to_datetime(purchase_requisitions["Requisition_Date"])

datasets = {
    "part_master": part_master,
    "equipment_models": equipment_models,
    "supplier_master": supplier_master,
    "demand_history": demand_history,
    "inventory_status": inventory_status,
    "purchase_requisitions": purchase_requisitions,
    "data_dictionary": data_dictionary
}

# Dataset Validation & Operational Data Review

This notebook validates the anonymised operational datasets used for the spare-parts warehouse analysis and automated warehouse transition project.

The objective is to confirm that the main inventory, warehouse, demand, supplier, and replenishment datasets are structurally consistent before starting the operational, inventory, and warehouse-automation analyses.

In [19]:
dataset_overview = pd.DataFrame({
    "Dataset": datasets.keys(),
    "Rows": [df.shape[0] for df in datasets.values()],
    "Columns": [df.shape[1] for df in datasets.values()]
})

dataset_overview

,Dataset,Rows,Columns
0,part_master,1000,40
1,equipment_models,24,4
2,supplier_master,20,7
3,demand_history,156000,5
4,inventory_status,1000,17
5,purchase_requisitions,47,8
6,data_dictionary,81,3


In [20]:
for name, df in datasets.items():
    print(f"\n{name.upper()} columns:")
    print(df.columns.tolist())


PART_MASTER columns:
['Part_ID', 'Part_Name', 'Equipment_Family_ID', 'Equipment_Family', 'Equipment_Model', 'Part_Category_ID', 'Part_Category', 'Demand_Profile', 'Criticality', 'Supplier_ID', 'Supplier_Region', 'Unit_Cost_EUR', 'Storage_Type', 'Lifecycle_Status', 'Target_Service_Level', 'Avg_Weekly_Demand_Seed', 'ABC_Class', 'Source_Context', 'Movement_Class', 'Movement_Lines_36M', 'Annual_Lines', 'Avg_Monthly_Lines', 'Cumulative_Movement_%', 'Manual_Location', 'UDC_Type', 'Dim_X_mm', 'Dim_Y_mm', 'Dim_Z_mm', 'Unit_Volume_cm3', 'Max_Dimension_mm', 'Min_Dimension_mm', 'Stock_Managed', 'Avg_Daily_Qty', 'Lead_Time_Weeks', 'Safety_Stock_Qty', 'Reorder_Point_Qty', 'Max_Stock_Qty', 'Stock_Qty', 'Total_Qty_36M', 'Total_Volume_cm3']

EQUIPMENT_MODELS columns:
['Equipment_Family_ID', 'Equipment_Family', 'Equipment_Model_ID', 'Equipment_Model']

SUPPLIER_MASTER columns:
['Supplier_ID', 'Supplier_Name', 'Supplier_Region', 'Avg_Lead_Time_Days', 'Lead_Time_SD_Days', 'Reliability_Score', 'Default_I

In [21]:
display(part_master.head())
display(inventory_status.head())
display(demand_history.head())

,Part_ID,Part_Name,Equipment_Family_ID,Equipment_Family,Equipment_Model,Part_Category_ID,Part_Category,Demand_Profile,Criticality,Supplier_ID,Supplier_Region,Unit_Cost_EUR,Storage_Type,Lifecycle_Status,Target_Service_Level,Avg_Weekly_Demand_Seed,ABC_Class,Source_Context,Movement_Class,Movement_Lines_36M,Annual_Lines,Avg_Monthly_Lines,Cumulative_Movement_%,Manual_Location,UDC_Type,Dim_X_mm,Dim_Y_mm,Dim_Z_mm,Unit_Volume_cm3,Max_Dimension_mm,Min_Dimension_mm,Stock_Managed,Avg_Daily_Qty,Lead_Time_Weeks,Safety_Stock_Qty,Reorder_Point_Qty,Max_Stock_Qty,Stock_Qty,Total_Qty_36M,Total_Volume_cm3
0,TC-CON-0343,Basic Line TC Grease Cartridge,TC,Tyre Changer,Basic Line TC,CON,Consumables & service kits,frequent,Medium,SUP-015,EU,40.87,Bin,Active,0.94,4.141,A,Anonymised/reconstructed warehouse transition ...,A+,473,157.7,13.14,7.47,R-I38,VER,1543,689,68,72293,1543,68,1,2.300,1.4,18,35,48,46,1795,3470064
1,VL-CON-0336,Scissor Lift Air Filter,VL,Vehicle Lift,Scissor Lift,CON,Consumables & service kits,frequent,Medium,SUP-004,EU,29.16,Bin,Active,0.94,4.364,A,Anonymised/reconstructed warehouse transition ...,A+,395,131.7,10.97,13.72,R-B17,D,258,91,20,470,258,20,1,1.706,1.0,13,22,38,23,1331,17860
2,TC-CON-0329,A2024 LL Air Filter,TC,Tyre Changer,A2024 LL,CON,Consumables & service kits,frequent,Medium,SUP-018,EU,43.54,Bin,Active,0.94,3.839,A,Anonymised/reconstructed warehouse transition ...,A+,313,104.3,8.69,18.66,R-E25,B,61,30,42,77,61,30,1,1.272,1.3,13,22,39,23,993,3003
3,WA-CON-0631,Exact Precision Lubrication Kit,WA,Wheel Aligner,Exact Precision,CON,Consumables & service kits,frequent,Medium,SUP-020,EU,26.96,Bin,Active,0.94,4.334,A,Anonymised/reconstructed warehouse transition ...,A+,231,77.0,6.42,22.31,R-F76,D,177,98,143,2480,177,98,1,0.867,1.3,15,21,32,16,677,79360
4,TC-CON-0171,Basic Line TC Air Filter,TC,Tyre Changer,Basic Line TC,CON,Consumables & service kits,frequent,Medium,SUP-003,EU,40.68,Bin,Active,0.94,4.441,A,Anonymised/reconstructed warehouse transition ...,A+,227,75.7,6.31,25.90,R-L86,B,135,132,46,820,135,46,1,1.030,1.0,6,12,20,17,804,16400


,Part_ID,On_Hand_Qty,On_Order_Qty,Allocated_Qty,Available_Qty,Avg_Weekly_Demand,Demand_SD,Demand_CV,Nonzero_Demand_Weeks,XYZ_Class,Lead_Time_Weeks,Safety_Stock_Qty,Reorder_Point_Qty,Suggested_Replenishment_Qty,Planner_Exception_Flag,Manual_Location,UDC_Type
0,TC-CON-0343,46,0,9,37,3.000000,3.270592,1.090197,102,Y,1.4,18,35,0,False,R-I38,VER
1,VL-CON-0336,23,0,7,16,3.852564,2.248453,0.583625,139,X,1.0,13,22,22,True,R-B17,D
2,TC-CON-0329,23,0,5,18,3.365385,2.032277,0.603877,135,X,1.3,13,22,21,True,R-E25,B
3,WA-CON-0631,16,7,4,19,3.602564,2.118062,0.587932,141,X,1.3,15,21,13,True,R-F76,D
4,TC-CON-0171,17,0,2,15,2.929487,3.428744,1.170425,95,Y,1.0,6,12,0,False,R-L86,B


,Week_Start,Part_ID,Demand_Qty,Order_Lines,Demand_Source
0,2022-01-03,TC-CON-0343,6,2,Anonymised operational history
1,2022-01-10,TC-CON-0343,3,1,Anonymised operational history
2,2022-01-17,TC-CON-0343,2,1,Anonymised operational history
3,2022-01-24,TC-CON-0343,0,0,Anonymised operational history
4,2022-01-31,TC-CON-0343,4,2,Anonymised operational history


## Structural Validation

The validation process checks:
- dataset completeness
- field consistency
- duplicate identifiers
- demand-history coverage
- inventory-control field availability
- relationships across inventory, supplier, and replenishment datasets

The objective is not aggressive data cleansing, but verification that the reconstructed datasets are operationally coherent and suitable for the following analyses.

In [22]:
missing_summary = []

for name, df in datasets.items():
    temp = pd.DataFrame({
        "Dataset": name,
        "Column": df.columns,
        "Missing_Values": df.isna().sum().values,
        "Missing_%": (df.isna().mean().values * 100).round(2)
    })
    missing_summary.append(temp)

missing_summary = pd.concat(missing_summary, ignore_index=True)

missing_summary[missing_summary["Missing_Values"] > 0].sort_values(
    ["Dataset", "Missing_Values"],
    ascending=[True, False]
)

,Dataset,Column,Missing_Values,Missing_%
63,inventory_status,Demand_CV,305,30.5


In [23]:
duplicate_checks = {
    "part_master.Part_ID": part_master["Part_ID"].duplicated().sum(),
    "equipment_models.Equipment_Model_ID": equipment_models["Equipment_Model_ID"].duplicated().sum(),
    "supplier_master.Supplier_ID": supplier_master["Supplier_ID"].duplicated().sum(),
    "inventory_status.Part_ID": inventory_status["Part_ID"].duplicated().sum(),
    "purchase_requisitions.Purchase_Requisition_ID": purchase_requisitions["Purchase_Requisition_ID"].duplicated().sum(),
    "demand_history.Part_ID + Week_Start": demand_history.duplicated(
        subset=["Part_ID", "Week_Start"]
    ).sum()
}

pd.DataFrame.from_dict(
    duplicate_checks,
    orient="index",
    columns=["Duplicate_Count"]
)

,Duplicate_Count
part_master.Part_ID,0
equipment_models.Equipment_Model_ID,0
supplier_master.Supplier_ID,0
inventory_status.Part_ID,0
purchase_requisitions.Purchase_Requisition_ID,0
demand_history.Part_ID + Week_Start,0


## Demand History Validation

Spare-parts demand is expected to be irregular, intermittent, and unevenly distributed across the warehouse population.

This section validates:
- demand-history coverage
- movement continuity
- intermittent demand behaviour
- inventory-demand consistency
- demand variability indicators used later in the ABC / XYZ and replenishment analyses.

In [24]:
demand_coverage = pd.DataFrame([{
    "Start_Date": demand_history["Week_Start"].min(),
    "End_Date": demand_history["Week_Start"].max(),
    "Weeks": demand_history["Week_Start"].nunique(),
    "Parts_in_Demand_History": demand_history["Part_ID"].nunique(),
    "Demand_Records": len(demand_history)
}])

demand_coverage

,Start_Date,End_Date,Weeks,Parts_in_Demand_History,Demand_Records
0,2022-01-03,2024-12-23,156,1000,156000


In [25]:
parts_in_master = set(part_master["Part_ID"])
parts_in_demand = set(demand_history["Part_ID"])

demand_part_coverage = pd.DataFrame({
    "Metric": [
        "Parts in part master",
        "Parts in demand history",
        "Parts without demand history",
        "Demand parts not in part master"
    ],
    "Count": [
        len(parts_in_master),
        len(parts_in_demand),
        len(parts_in_master - parts_in_demand),
        len(parts_in_demand - parts_in_master)
    ]
})

demand_part_coverage

,Metric,Count
0,Parts in part master,1000
1,Parts in demand history,1000
2,Parts without demand history,0
3,Demand parts not in part master,0


In [26]:
demand_by_part = (
    demand_history
    .groupby("Part_ID")
    .agg(
        Total_Demand=("Demand_Qty", "sum"),
        Avg_Weekly_Demand=("Demand_Qty", "mean"),
        Demand_Std=("Demand_Qty", "std"),
        Weeks_With_Demand=("Demand_Qty", lambda x: (x > 0).sum()),
        Total_Weeks=("Demand_Qty", "count")
    )
    .reset_index()
)

demand_by_part["Zero_Demand_Weeks"] = (
    demand_by_part["Total_Weeks"] - demand_by_part["Weeks_With_Demand"]
)

demand_by_part["Zero_Demand_%"] = (
    demand_by_part["Zero_Demand_Weeks"] / demand_by_part["Total_Weeks"] * 100
).round(2)

demand_by_part["Coefficient_of_Variation"] = np.where(
    demand_by_part["Avg_Weekly_Demand"] > 0,
    demand_by_part["Demand_Std"] / demand_by_part["Avg_Weekly_Demand"],
    np.nan
)

demand_by_part.describe().round(2)

,Total_Demand,Avg_Weekly_Demand,Demand_Std,Weeks_With_Demand,Total_Weeks,Zero_Demand_Weeks,Zero_Demand_%,Coefficient_of_Variation
count,1000.00,1000.00,1000.00,1000.00,1000.0,1000.00,1000.00,695.00
mean,35.64,0.23,0.50,14.95,156.0,141.05,90.42,5.05
std,68.97,0.44,0.55,23.25,0.0,23.25,14.90,3.18
min,0.00,0.00,0.00,0.00,156.0,15.00,9.62,0.58
25%,0.00,0.00,0.00,0.00,156.0,136.00,87.18,2.43
50%,9.00,0.06,0.34,5.00,156.0,151.00,96.79,4.66
75%,39.00,0.25,0.80,20.00,156.0,156.00,100.00,6.55
max,601.00,3.85,3.43,141.00,156.0,156.00,100.00,12.49


In [27]:
intermittency_summary = pd.DataFrame({
    "Metric": [
        "Parts analysed",
        "Parts with zero total demand",
        "Parts with 75%+ zero-demand weeks",
        "Average zero-demand weeks %"
    ],
    "Value": [
        len(demand_by_part),
        (demand_by_part["Total_Demand"] == 0).sum(),
        (demand_by_part["Zero_Demand_%"] >= 75).sum(),
        round(demand_by_part["Zero_Demand_%"].mean(), 2)
    ]
})

intermittency_summary

,Metric,Value
0,Parts analysed,1000.00
1,Parts with zero total demand,305.00
2,Parts with 75%+ zero-demand weeks,864.00
3,Average zero-demand weeks %,90.42


## Inventory and Replenishment Validation

This section validates the inventory-control and replenishment fields required for the operational and warehouse analyses.

The checks focus on:
- stock availability
- reorder-point consistency
- safety-stock indicators
- replenishment exceptions
- purchase requisition availability
- operational inventory-control indicators

In [28]:
inventory_status.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Part_ID,1000,1000,TC-CON-0343,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
On_Hand_Qty,1000.0,NaN,NaN,NaN,5.422,4.847826,1.0,2.0,4.0,7.0,46.0
On_Order_Qty,1000.0,NaN,NaN,NaN,0.792,1.765745,0.0,0.0,0.0,0.0,18.0
Allocated_Qty,1000.0,NaN,NaN,NaN,0.18,0.604951,0.0,0.0,0.0,0.0,9.0
Available_Qty,1000.0,NaN,NaN,NaN,6.034,4.906416,1.0,3.0,4.0,7.0,37.0
Avg_Weekly_Demand,1000.0,NaN,NaN,NaN,0.228429,0.442127,0.0,0.0,0.057692,0.25,3.852564
Demand_SD,1000.0,NaN,NaN,NaN,0.502677,0.554679,0.0,0.0,0.344096,0.800001,3.428744
Demand_CV,695.0,NaN,NaN,NaN,5.049941,3.180196,0.583625,2.433115,4.658661,6.554033,12.489996
Nonzero_Demand_Weeks,1000.0,NaN,NaN,NaN,14.951,23.248593,0.0,0.0,5.0,20.0,141.0
XYZ_Class,1000,3,Z,820,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
inventory_checks = {
    "Negative on-hand quantity": (inventory_status["On_Hand_Qty"] < 0).sum(),
    "Negative on-order quantity": (inventory_status["On_Order_Qty"] < 0).sum(),
    "Negative allocated quantity": (inventory_status["Allocated_Qty"] < 0).sum(),
    "Negative available quantity": (inventory_status["Available_Qty"] < 0).sum(),
    "Negative safety stock": (inventory_status["Safety_Stock_Qty"] < 0).sum(),
    "Negative reorder point": (inventory_status["Reorder_Point_Qty"] < 0).sum(),
    "Parts below reorder point": (
        inventory_status["Available_Qty"] < inventory_status["Reorder_Point_Qty"]
    ).sum(),
    "Planner exceptions": (
        inventory_status["Planner_Exception_Flag"] == True
    ).sum()
}

pd.DataFrame.from_dict(
    inventory_checks,
    orient="index",
    columns=["Count"]
)

,Count
Negative on-hand quantity,0
Negative on-order quantity,0
Negative allocated quantity,0
Negative available quantity,0
Negative safety stock,0
Negative reorder point,0
Parts below reorder point,47
Planner exceptions,47


In [30]:
#purchase_requisitions.head()

In [31]:
purchase_requisition_summary = pd.DataFrame({
    "Metric": [
        "Purchase requisition records",
        "Unique parts with requisitions",
        "Unique suppliers",
        "Total requested quantity",
        "Total estimated value EUR"
    ],
    "Value": [
        len(purchase_requisitions),
        purchase_requisitions["Part_ID"].nunique(),
        purchase_requisitions["Supplier_ID"].nunique(),
        purchase_requisitions["Requested_Qty"].sum(),
        purchase_requisitions["Estimated_Value_EUR"].sum()
    ]
})

purchase_requisition_summary

,Metric,Value
0,Purchase requisition records,47.00
1,Unique parts with requisitions,47.00
2,Unique suppliers,19.00
3,Total requested quantity,303.00
4,Total estimated value EUR,15003.97


## Segmentation and Supplier Validation

Inventory classifications and supplier structures are important because they support:
- inventory prioritisation
- replenishment analysis
- warehouse movement evaluation
- operational risk monitoring
- warehouse automation feasibility analyses

This section validates the availability and consistency of:
- ABC classifications
- XYZ classifications
- supplier lead-time indicators
- supplier-region relationships

In [32]:
part_with_xyz = part_master.merge(
    inventory_status[["Part_ID", "XYZ_Class"]],
    on="Part_ID",
    how="left"
)

abc_xyz_summary = pd.crosstab(
    part_with_xyz["ABC_Class"],
    part_with_xyz["XYZ_Class"],
    margins=True
)

abc_xyz_summary

XYZ_Class,X,Y,Z,All
ABC_Class,,,,
A,8,63,378,449
B,3,50,242,295
C,0,56,200,256
All,11,169,820,1000


In [33]:
supplier_summary = supplier_master.copy()

supplier_summary

,Supplier_ID,Supplier_Name,Supplier_Region,Avg_Lead_Time_Days,Lead_Time_SD_Days,Reliability_Score,Default_Incoterm
0,SUP-001,Italy Industrial Components 01,Italy,4,1,0.86,DAP
1,SUP-002,Italy Industrial Components 02,Italy,4,1,0.89,EXW
2,SUP-003,EU Industrial Components 03,EU,6,1,0.85,DAP
3,SUP-004,EU Industrial Components 04,EU,7,3,0.95,EXW
4,SUP-005,Eastern Europe Industrial Components 05,Eastern Europe,10,1,0.75,DDP
5,SUP-006,EU Industrial Components 06,EU,7,2,0.96,DAP
6,SUP-007,Eastern Europe Industrial Components 07,Eastern Europe,10,1,0.91,FCA
7,SUP-008,EU Industrial Components 08,EU,8,1,0.96,DAP
8,SUP-009,Italy Industrial Components 09,Italy,4,1,0.87,EXW
9,SUP-010,EU Industrial Components 10,EU,8,2,0.92,FCA


In [34]:
part_supplier_summary = (
    part_master
    .groupby("Supplier_Region")
    .agg(
        Parts=("Part_ID", "count"),
        Avg_Unit_Cost_EUR=("Unit_Cost_EUR", "mean"),
        Avg_Target_Service_Level=("Target_Service_Level", "mean")
    ).round(2)
    .reset_index()
    .sort_values("Parts", ascending=False)
)

part_supplier_summary

,Supplier_Region,Parts,Avg_Unit_Cost_EUR,Avg_Target_Service_Level
0,EU,604,218.16,0.95
2,Italy,180,243.77,0.95
1,Eastern Europe,154,196.93,0.95
3,North America,62,181.25,0.95


## Validation Summary

The operational datasets are structurally consistent and suitable for the following analyses.

The validation confirmed:
- consistency across the main inventory, demand, supplier, and replenishment datasets
- full demand-history coverage across the spare-parts population
- realistic intermittent demand behaviour typical of spare-parts environments
- availability of inventory-control and replenishment indicators
- consistency of the ERP/WMS-style operational relationships across datasets
- availability of movement, inventory, and warehouse-related attributes required for the warehouse-transition analyses

The next analyses will focus on:
- inventory classification and segmentation
- inventory concentration
- movement behaviour
- warehouse operational structure
- automated warehouse feasibility evaluation